# Four focused Local SGD experiments on MNIST

This notebook contains experiment design and orchestration only. Training, models, controlled heterogeneity, FedAvg, and drift come from `src/`; visual styling comes from the project's plot templates. The four questions are: unequal local work for a hard client, number of clients, communication-round weighting, and partial participation.

Each condition is paired over seeds and reported as mean ±1 standard deviation. Training loss is the primary optimization metric. Curves use both communication rounds and actual client mini-batch gradient updates. See `notebookREADME.md` for the full controls and interpretation rules.

In [ ]:
from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    matches = [p for p in [ROOT, *ROOT.parents] if (p / 'src').is_dir()]
    if not matches:
        raise RuntimeError('Open this notebook from the Shalilious repository.')
    ROOT = matches[0]
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))

from data import DATASETS, DeterministicGaussianNoise
from studies import make_partition, run_federated
from plots.notebook_experiments import budget_curve_plot, hard_client_plot

print('Project:', ROOT)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Configuration and shared data

The default profile uses half of MNIST, three paired seeds, and about 60,000 local optimizer steps across all four studies. It is designed for a roughly 90–120 minute P100 session, though runtimes vary. Set `PILOT=True` to validate the full pipeline in minutes. Set `DATA_FRAC=1.0` afterward if full MNIST is required.

Drift is measured only at initialization and the final broadcast because it requires exact gradients over every client shard. Those diagnostic passes are deliberately excluded from the training budget.

In [ ]:
PILOT = False
SEEDS = [0] if PILOT else [0, 1, 2]
DATA_FRAC = 0.10 if PILOT else 0.50
DATA_POOL_SEED = 2026
MODEL = 'small_cnn'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if not PILOT and DEVICE != 'cuda':
    raise RuntimeError('The full profile requires a CUDA GPU; use a P100 or set PILOT=True.')
BATCH_SIZE = 64
LEARNING_RATE = 0.05
BASE_ROUNDS = 4 if PILOT else 30
EVAL_EVERY = 1 if PILOT else 5

RUN_HARD_CLIENT = True
RUN_CLIENT_COUNT = True
RUN_AGGREGATION = True
RUN_PARTICIPATION = True

DATA_ROOT = ROOT / 'data'
OUTPUT_ROOT = ROOT / 'notebook_results'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
dataset, pool, labels = DATASETS['mnist']['load'](
    str(DATA_ROOT), data_frac=DATA_FRAC, seed=DATA_POOL_SEED
)

CONFIG = {
    'pilot': PILOT, 'seeds': SEEDS, 'data_frac': DATA_FRAC,
    'data_pool_seed': DATA_POOL_SEED, 'model': MODEL, 'device': DEVICE,
    'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE,
    'base_rounds': BASE_ROUNDS, 'eval_every': EVAL_EVERY,
}
(OUTPUT_ROOT / 'config.json').write_text(json.dumps(CONFIG, indent=2))
print(f'Loaded {len(pool):,} stratified MNIST examples on {DEVICE}.')

In [ ]:
def tagged_history(history, **metadata):
    return [{**row, **metadata} for row in history]


def _csv_value(value):
    if isinstance(value, (tuple, list, dict)):
        return json.dumps(value)
    return value


def save_study(name, histories, summaries, figure):
    out = OUTPUT_ROOT / name
    out.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([{k: _csv_value(v) for k, v in row.items()} for row in histories]).to_csv(
        out / 'history.csv', index=False
    )
    summary_frame = pd.DataFrame([
        {k: _csv_value(v) for k, v in row.items()} for row in summaries
    ])
    summary_frame.to_csv(out / 'summary.csv', index=False)
    figure.savefig(out / f'{name}.png', dpi=160, facecolor='#fcfcfb')
    display(summary_frame)
    display(figure)
    plt.close(figure)
    print('Saved:', out)
    return summary_frame

## 1. Two clients: vary the hard client's local steps

Both clients receive IID, near-equal shards. Client 1 alone gets fixed Gaussian corruption after MNIST normalization. `K_easy=5` stays fixed and `K_hard` changes. This isolates a compute-allocation question under covariate shift; label heterogeneity should remain near zero. Increasing `K_hard` also increases total work, so the saved update count is part of the result.

In [ ]:
if RUN_HARD_CLIENT:
    EASY_K = 5
    HARD_K_VALUES = [1, 3, 5, 8]
    NOISE_STD = 0.8
    hard_histories, hard_summaries = [], []
    for seed in SEEDS:
        shards = make_partition(labels, pool, 2, seed, partition='iid')
        hard_dataset = DeterministicGaussianNoise(dataset, std=NOISE_STD, seed=100_000 + seed)
        for hard_k in HARD_K_VALUES:
            print(f'[hard client] seed={seed} K_hard={hard_k}')
            history, summary = run_federated(
                dataset, pool, labels, n_clients=2, seed=seed,
                rounds=BASE_ROUNDS, local_steps=[EASY_K, hard_k],
                partition='iid', shards=shards,
                client_datasets=[dataset, hard_dataset], aggregation='data',
                model=MODEL, batch_size=BATCH_SIZE, lr=LEARNING_RATE,
                device=DEVICE, eval_every=EVAL_EVERY,
            )
            summary.update({'easy_k': EASY_K, 'hard_k': hard_k, 'noise_std': NOISE_STD})
            hard_summaries.append(summary)
            hard_histories.extend(tagged_history(history, seed=seed, hard_k=hard_k))
    hard_summary = save_study(
        '01_hard_client', hard_histories, hard_summaries, hard_client_plot(hard_summaries)
    )

## 2. Number of clients

Full participation, shared `K=5`, model, learning rate, and round budget stay fixed. We compare `{2,5,10}` clients at controlled `H=0` and common feasible `H=0.25`. The source partitioner both constructs and measures heterogeneity; plots never substitute requested H for realized H. More clients mean more work per round, hence the paired communication/computation panels.

In [ ]:
if RUN_CLIENT_COUNT:
    CLIENT_COUNTS = [2, 5, 10]
    CLIENT_H = {'H=0.00': 0.0, 'H=0.25': 0.25}
    count_histories, count_summaries = [], []
    for condition, target_h in CLIENT_H.items():
        for n_clients in CLIENT_COUNTS:
            for seed in SEEDS:
                print(f'[client count] {condition} N={n_clients} seed={seed}')
                history, summary = run_federated(
                    dataset, pool, labels, n_clients=n_clients, seed=seed,
                    rounds=BASE_ROUNDS, local_steps=5, partition='target_h',
                    target_h=target_h, aggregation='data', model=MODEL,
                    batch_size=BATCH_SIZE, lr=LEARNING_RATE, device=DEVICE,
                    eval_every=EVAL_EVERY,
                )
                summary['condition'] = condition
                count_summaries.append(summary)
                count_histories.extend(tagged_history(
                    history, seed=seed, client_count=n_clients, condition=condition
                ))
    figure = budget_curve_plot(
        count_histories, group_key='client_count', group_order=CLIENT_COUNTS,
        group_labels={n: f'{n} clients' for n in CLIENT_COUNTS},
        facet_key='condition', facet_order=list(CLIENT_H),
        facet_labels={k: k for k in CLIENT_H}, title='Effect of federation size',
    )
    count_summary = save_study('02_client_count', count_histories, count_summaries, figure)

## 3. Weighting client endpoints at communication

A seed-specific Dirichlet split (`alpha=0.3`) is reused exactly across policies so unequal quantities make uniform and data weighting distinct. All policies are judged against the same data-weighted global objective. `inverse_js` is exploratory and can introduce objective bias. FedNova is omitted here because equal K makes it identical to data-weighted FedAvg.

In [ ]:
if RUN_AGGREGATION:
    AGGREGATIONS = ['uniform', 'data', 'inverse_js']
    DIRICHLET_ALPHA = 0.3
    INVERSE_JS_BETA = 2.0
    aggregation_histories, aggregation_summaries = [], []
    for seed in SEEDS:
        shards = make_partition(
            labels, pool, 10, seed, partition='dirichlet', alpha=DIRICHLET_ALPHA
        )
        for policy in AGGREGATIONS:
            print(f'[aggregation] seed={seed} policy={policy}')
            history, summary = run_federated(
                dataset, pool, labels, n_clients=10, seed=seed,
                rounds=BASE_ROUNDS, local_steps=5, partition='dirichlet',
                alpha=DIRICHLET_ALPHA, shards=shards, aggregation=policy,
                aggregation_beta=INVERSE_JS_BETA, model=MODEL,
                batch_size=BATCH_SIZE, lr=LEARNING_RATE, device=DEVICE,
                eval_every=EVAL_EVERY,
            )
            aggregation_summaries.append(summary)
            aggregation_histories.extend(tagged_history(
                history, seed=seed, aggregation_policy=policy
            ))
    figure = budget_curve_plot(
        aggregation_histories, group_key='aggregation_policy',
        group_order=AGGREGATIONS, group_labels={p: p for p in AGGREGATIONS},
        title='Communication-round weighting on unequal client shards',
    )
    aggregation_summary = save_study(
        '03_aggregation', aggregation_histories, aggregation_summaries, figure
    )

## 4. Fraction of clients participating

Ten total clients are fixed. Each round samples `ceil(qN)` without replacement at `q in {0.2,0.5,1.0}`. We repeat at controlled `H=0` and `H=0.8`. Smaller fractions get proportionally more rounds, so every curve ends at the same local-update budget as 30 full-participation rounds. This lets the left panels show communication cost while the right panels make the compute-matched comparison.

In [ ]:
if RUN_PARTICIPATION:
    TOTAL_CLIENTS = 10
    PARTICIPATION_FRACTIONS = [0.2, 0.5, 1.0]
    PARTICIPATION_H = {'H=0.00': 0.0, 'H=0.80': 0.8}
    participation_histories, participation_summaries = [], []
    for condition, target_h in PARTICIPATION_H.items():
        for fraction in PARTICIPATION_FRACTIONS:
            online = math.ceil(fraction * TOTAL_CLIENTS)
            rounds = math.ceil(BASE_ROUNDS * TOTAL_CLIENTS / online)
            eval_every = max(1, rounds // 6)
            for seed in SEEDS:
                print(f'[participation] {condition} q={fraction} seed={seed} rounds={rounds}')
                history, summary = run_federated(
                    dataset, pool, labels, n_clients=TOTAL_CLIENTS, seed=seed,
                    rounds=rounds, local_steps=5, partition='target_h',
                    target_h=target_h, aggregation='data',
                    client_fraction=fraction, model=MODEL, batch_size=BATCH_SIZE,
                    lr=LEARNING_RATE, device=DEVICE, eval_every=eval_every,
                )
                summary['condition'] = condition
                participation_summaries.append(summary)
                participation_histories.extend(tagged_history(
                    history, seed=seed, participation_fraction=fraction,
                    condition=condition
                ))
    figure = budget_curve_plot(
        participation_histories, group_key='participation_fraction',
        group_order=PARTICIPATION_FRACTIONS,
        group_labels={q: f'q={q:g}' for q in PARTICIPATION_FRACTIONS},
        facet_key='condition', facet_order=list(PARTICIPATION_H),
        facet_labels={k: k for k in PARTICIPATION_H},
        title='Partial participation at a matched computation budget',
    )
    participation_summary = save_study(
        '04_participation', participation_histories, participation_summaries, figure
    )

## Interpretation checklist

Use paired-seed differences and uncertainty, not the best seed. Check `h_label` and `quantity_cv` before interpreting any heterogeneous comparison. For participation, report requested and actual fractions plus `participation_coverage` and `participation_cv`. Treat inverse-JS as a biased alternative objective weight, not a guaranteed correction. The exact gradient drift describes gradient dissimilarity at a common broadcast point; it is not endpoint distance and it is not the running minimum gradient norm.